# Policy rollout and replay checks

This notebook loads a trained vision-only policy and tests it in the LIBERO simulator. It also replays recorded actions to help distinguish policy errors from simulator problems.

Run the cells from top to bottom. The scenes used here come from training episodes and are for development, not final held-out evaluation. The simulator success check determines task completion; prediction loss alone does not.


## 1. Load the saved training run

Import the libraries, choose the saved run, and load its checkpoint and data manifest. The fingerprint check confirms that the checkpoint uses the expected data preparation. No training happens here.


In [ ]:
from pathlib import Path
import sys
import json
import hashlib
import xml.etree.ElementTree as ET

# Preserve the import order that worked during setup.
import torch
import torchvision
from torch import nn

import numpy as np
import h5py
from PIL import Image
from IPython.display import display, Video

project_dir = Path.home() / "t2-project"

sys.path.insert(0, str(project_dir / "LIBERO"))

import robosuite
import robosuite.macros as macros

# Match the image convention recorded in your demonstrations.
macros.IMAGE_CONVENTION = "opengl"

from libero.libero import benchmark, get_libero_path
from libero.libero.envs import OffScreenRenderEnv

run_dir = (
    project_dir
    / "runs"
    / "vision_only_seed0_20261117T204733_783686Z"
)

checkpoint_path = run_dir / "best_model.pt"

assert checkpoint_path.is_file(), checkpoint_path

manifest_bytes = (run_dir / "split_manifest.json").read_bytes()
manifest = json.loads(manifest_bytes)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True
)

assert hashlib.sha256(manifest_bytes).hexdigest() == (
    checkpoint["config"]["manifest_sha256"]
)

camera_names = checkpoint["config"]["cameras"]

print("Device:", device)
print("Checkpoint epoch:", checkpoint["epoch"])
print("Validation MSE:", checkpoint["validation_loss"])

## 2. Restore the trained policy

Recreate the same model architecture used during training and load its learned weights. Evaluation mode prepares the model for prediction. The saved action mean and scale will convert predictions back to robot commands.


In [ ]:
class VisionEncoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.convolutions = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),

            nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),

            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
        )

    def forward(self, images):
        return self.convolutions(images)


class VisionOnlyPolicy(nn.Module):
    def __init__(self, camera_names):
        super().__init__()

        self.camera_names = list(camera_names)
        self.encoder = VisionEncoder()
        self.pool = nn.AdaptiveAvgPool2d((4, 4))

        combined_features = len(self.camera_names) * 64 * 4 * 4

        self.action_head = nn.Sequential(
            nn.Linear(combined_features, 256),
            nn.ReLU(),
            nn.Linear(256, 7),
        )

    def forward(self, images):
        camera_features = []

        for camera in self.camera_names:
            feature_map = self.encoder(images[camera])
            pooled = self.pool(feature_map)
            camera_features.append(pooled.flatten(start_dim=1))

        combined = torch.cat(camera_features, dim=1)

        return self.action_head(combined)


policy = VisionOnlyPolicy(camera_names).to(device)

policy.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

policy.eval()

action_mean = np.asarray(
    checkpoint["normalization"]["mean"],
    dtype=np.float32
)

action_scale = np.asarray(
    checkpoint["normalization"]["scale"],
    dtype=np.float32
)

print("Trained policy loaded.")

## 3. Choose a task and restore its scene

Choose bowl-on-plate or bowl-on-stove and a training episode for development testing. Prepare its saved scene, local asset paths, and simulator state. This is a training-scene diagnostic, not held-out evaluation. To switch tasks, change task_name here and rerun this cell and all cells below it.


In [ ]:
task_name = "put_the_bowl_on_the_stove"
rollout_seed = 0
max_steps = 300

# Use a training episode for integration debugging.
episode_name = manifest["splits"]["train"][task_name][0]

dataset_path = (
    project_dir
    / manifest["tasks"][task_name]["file"]
)

with h5py.File(dataset_path, "r") as demo_file:
    episode = demo_file["data"][episode_name]

    model_xml = episode.attrs["model_file"]

    if isinstance(model_xml, bytes):
        model_xml = model_xml.decode("utf-8")

    # states[1] corresponds to the first saved camera observation.
    development_state = episode["states"][1].copy()

    reference_images = {
        camera: episode["obs"][camera][0]
        for camera in camera_names
    }

    reference_joints = episode["obs"]["joint_states"][0]

# Rewrite asset paths from the dataset creator's computer to yours.
xml_root = ET.fromstring(model_xml)

robosuite_dir = Path(robosuite.__file__).resolve().parent
libero_assets_dir = Path(get_libero_path("assets"))

for element in xml_root.findall("./asset/*"):
    old_path = element.get("file")

    if old_path is None:
        continue

    if "/robosuite/" in old_path:
        suffix = old_path.rsplit("/robosuite/", 1)[1]
        local_path = robosuite_dir / suffix

    elif "/assets/" in old_path:
        suffix = old_path.split("/assets/", 1)[1]
        local_path = libero_assets_dir / suffix

    else:
        local_path = Path(old_path)

    local_path = local_path.resolve()

    if not local_path.is_file():
        raise FileNotFoundError(
            f"Cannot resolve simulator asset: {old_path}\n"
            f"Tried: {local_path}"
        )

    element.set("file", str(local_path))

local_model_xml = ET.tostring(
    xml_root,
    encoding="unicode"
)

suite = benchmark.get_benchmark_dict()["libero_goal"]()

task = next(
    suite.get_task(i)
    for i in range(suite.get_num_tasks())
    if suite.get_task(i).name == task_name
)

bddl_path = (
    Path(get_libero_path("bddl_files"))
    / task.problem_folder
    / task.bddl_file
)

print("Task:", task_name)
print("Development episode:", episode_name)
print("Seed:", rollout_seed)

## 4. Convert camera observations into actions

Map the simulator camera names to the model inputs and apply the same image preprocessing used in training. Predict one action and undo action normalization. Preserve the stored image orientation for the model; flip images only for display.


In [ ]:
camera_mapping = {
    "agentview_rgb": "agentview_image",
    "eye_in_hand_rgb": "robot0_eye_in_hand_image",
}


def predict_action(observation):
    model_images = {}

    for dataset_camera in camera_names:
        simulator_camera = camera_mapping[dataset_camera]

        image = observation[simulator_camera]

        assert image.shape == (128, 128, 3)

        image = image.astype(np.float32) / 255.0
        image = np.ascontiguousarray(
            image.transpose(2, 0, 1)
        )

        model_images[dataset_camera] = (
            torch.from_numpy(image)
            .unsqueeze(0)
            .to(device)
        )

    with torch.no_grad():
        normalized_action = policy(model_images)[0].cpu().numpy()

    raw_action = normalized_action * action_scale + action_mean

    assert raw_action.shape == (7,)
    assert np.isfinite(raw_action).all()

    return raw_action

## 5. Run the learned policy

Create the simulator, restore the scene, and check the cameras and robot state. Repeatedly predict an action, clip it to the simulator bounds, execute it, and read the next observation. Stop on simulator success, environment termination, or the 300-step limit. The simulator closes afterward.


In [ ]:
env = None
frames = []
rollout_result = None

try:
    print("Creating simulator...", flush=True)

    env = OffScreenRenderEnv(
        bddl_file_name=str(bddl_path),
        camera_heights=128,
        camera_widths=128,
        control_freq=20,
        horizon=max_steps,
        ignore_done=False,
    )

    env.seed(rollout_seed)
    env.reset()

    env.reset_from_xml_string(local_model_xml)
    env.sim.reset()

    observation = env.set_init_state(development_state)

    # Confirm robot-state restoration.
    assert np.allclose(
        observation["robot0_joint_pos"],
        reference_joints,
        rtol=0,
        atol=1e-6
    )

    print("Saved robot state restored.", flush=True)

    # Compare stored and newly rendered images before policy execution.
    for dataset_camera, simulator_camera in camera_mapping.items():
        stored = reference_images[dataset_camera].astype(np.float32)
        live = observation[simulator_camera].astype(np.float32)

        direct_error = np.abs(stored - live).mean()
        flipped_error = np.abs(stored - live[::-1]).mean()

        print(
            f"{dataset_camera}: "
            f"direct pixel difference={direct_error:.2f}, "
            f"flipped difference={flipped_error:.2f}",
            flush=True
        )

        assert direct_error <= flipped_error, (
            "Camera orientation may differ from training."
        )

    # Show the external camera: saved image left, live image right.
    comparison = np.concatenate(
        [
            reference_images["agentview_rgb"][::-1],
            observation["agentview_image"][::-1],
        ],
        axis=1
    )

    display(Image.fromarray(comparison))

    action_low, action_high = env.env.action_spec
    action_low = np.asarray(action_low)
    action_high = np.asarray(action_high)

    assert action_low.shape == (7,)
    assert action_high.shape == (7,)

    print("Action bounds:", action_low, action_high, flush=True)

    # Avoid counting a scene that is already successful.
    assert not env.check_success(), (
        "The restored scene already satisfies the task."
    )

    frames.append(observation["agentview_image"][::-1].copy())

    success = False
    clipped_steps = 0
    steps_taken = 0
    termination = "timeout"

    for step in range(max_steps):
        raw_action = predict_action(observation)

        # Restrict the prediction to this simulator's valid command range.
        action = np.clip(raw_action, action_low, action_high)

        if np.any(action != raw_action):
            clipped_steps += 1

        observation, reward, done, info = env.step(action.tolist())

        steps_taken = step + 1
        success = bool(env.check_success())

        frames.append(observation["agentview_image"][::-1].copy())

        if steps_taken % 25 == 0:
            print(
                f"Step {steps_taken}/{max_steps}"
                f" | success={success}",
                flush=True
            )

        if success:
            termination = "success"
            break

        if done:
            termination = "environment_done"
            break

    rollout_result = {
        "purpose": "development rollout; training-scene start",
        "task": task_name,
        "source_episode": episode_name,
        "source_state_index": 1,
        "seed": rollout_seed,
        "checkpoint_epoch": checkpoint["epoch"],
        "checkpoint_file": str(checkpoint_path.relative_to(project_dir)),
        "success": success,
        "steps": steps_taken,
        "timeout_steps": max_steps,
        "termination": termination,
        "steps_with_clipped_actions": clipped_steps,
        "action_low": action_low.tolist(),
        "action_high": action_high.tolist(),
    }

    print(json.dumps(rollout_result, indent=2))

finally:
    if env is not None:
        env.close()

    print("Simulator closed.", flush=True)

## 6. Save the policy rollout

Save the development result as JSON and the camera frames as an MP4. These commands came from the learned policy. This development video is not a held-out success-rate measurement or the complete assignment execution recording.


In [ ]:
from datetime import datetime, timezone
import imageio.v2 as imageio

assert rollout_result is not None, (
    "Complete the rollout cell successfully first."
)

stamp = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S_%fZ"
)

rollout_dir = run_dir / "development_rollouts"
rollout_dir.mkdir(parents=True, exist_ok=True)

file_stem = f"{task_name}_seed{rollout_seed}_{stamp}"

result_path = rollout_dir / f"{file_stem}.json"
video_path = rollout_dir / f"{file_stem}.mp4"

result_path.write_text(
    json.dumps(rollout_result, indent=2),
    encoding="utf-8"
)

with imageio.get_writer(
    str(video_path),
    fps=20
) as writer:
    for frame in frames:
        writer.append_data(frame)

print("Development rollout — not held-out evaluation")
print("Task:", task_name)
print("Seed:", rollout_seed)
print("Simulator success:", rollout_result["success"])
print("Saved video:", video_path)

display(Video(filename=str(video_path), embed=True))

## 7. Load recorded demonstration actions

Load the demonstrator's commands for a replay diagnostic. Start with action 1 because the restored scene is state 1. Recorded replay does not use the learned policy.


In [ ]:
with h5py.File(dataset_path, "r") as demo_file:
    episode = demo_file["data"][episode_name]

    replay_actions = episode["actions"][1:].copy()
    replay_states = episode["states"][:]

print("Task:", task_name)
print("Episode:", episode_name)
print("Recorded actions to replay:", len(replay_actions))

## 8. Replay and check the simulator

Execute the recorded actions from the same development scene. Check task success and compare resulting simulator states with saved states. Successful replay shows that these commands can complete this scene; nonzero state errors mean the trajectory was not reproduced exactly.


In [ ]:
replay_env = None
replay_frames = []
replay_result = None

try:
    replay_env = OffScreenRenderEnv(
        bddl_file_name=str(bddl_path),
        camera_heights=128,
        camera_widths=128,
        control_freq=20,
        horizon=max_steps,
        ignore_done=False,
    )

    replay_env.seed(rollout_seed)
    replay_env.reset()

    replay_env.reset_from_xml_string(local_model_xml)
    replay_env.sim.reset()

    observation = replay_env.set_init_state(development_state)

    assert not replay_env.check_success(), (
        "The restored scene already satisfies the task."
    )

    replay_frames.append(
        observation["agentview_image"][::-1].copy()
    )

    low, high = replay_env.env.action_spec

    state_errors = []
    success = False
    steps_taken = 0
    termination = "recorded_actions_exhausted"

    for action_index, action in enumerate(replay_actions, start=1):
        if steps_taken >= max_steps:
            termination = "timeout"
            break

        assert np.isfinite(action).all()

        # Do not silently alter the recorded demonstration commands.
        assert np.all(action >= np.asarray(low) - 1e-6)
        assert np.all(action <= np.asarray(high) + 1e-6)

        observation, reward, done, info = replay_env.step(
            action.tolist()
        )

        steps_taken += 1

        replay_frames.append(
            observation["agentview_image"][::-1].copy()
        )

        # After actions[k], compare against states[k + 1].
        expected_state_index = action_index + 1

        if expected_state_index < len(replay_states):
            actual_state = replay_env.get_sim_state()
            expected_state = replay_states[expected_state_index]

            assert actual_state.shape == expected_state.shape

            # Exclude the simulation timestamp.
            error = np.linalg.norm(
                actual_state[1:] - expected_state[1:]
            )

            state_errors.append(float(error))

        success = bool(replay_env.check_success())

        if steps_taken % 25 == 0:
            print(
                f"Replay step {steps_taken}"
                f" | success={success}",
                flush=True
            )

        if success:
            termination = "success"
            break

        if done:
            termination = "environment_done"
            break

    replay_result = {
        "purpose": "recorded-action replay; not learned policy",
        "task": task_name,
        "source_episode": episode_name,
        "source_state_index": 1,
        "seed": rollout_seed,
        "success": success,
        "steps": steps_taken,
        "termination": termination,
        "state_comparisons": len(state_errors),
        "mean_state_error": (
            float(np.mean(state_errors))
            if state_errors else None
        ),
        "max_state_error": (
            float(np.max(state_errors))
            if state_errors else None
        ),
    }

    print(json.dumps(replay_result, indent=2))

finally:
    if replay_env is not None:
        replay_env.close()

    print("Replay simulator closed.", flush=True)

## 9. Save the recorded replay

Save a separate replay result and video. Keep these labeled as recorded demonstration replay so they cannot be confused with autonomous learned-policy performance.


In [ ]:
assert replay_result is not None

stamp = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S_%fZ"
)

replay_dir = run_dir / "replay_diagnostics"
replay_dir.mkdir(parents=True, exist_ok=True)

file_stem = f"recorded_replay_{task_name}_{stamp}"

replay_json_path = replay_dir / f"{file_stem}.json"
replay_video_path = replay_dir / f"{file_stem}.mp4"

replay_json_path.write_text(
    json.dumps(replay_result, indent=2),
    encoding="utf-8"
)

with imageio.get_writer(
    str(replay_video_path),
    fps=20
) as writer:
    for frame in replay_frames:
        writer.append_data(frame)

print("RECORDED DEMONSTRATION REPLAY — NOT LEARNED POLICY")
print("Task:", task_name)
print("Success:", replay_result["success"])

display(
    Video(
        filename=str(replay_video_path),
        embed=True
    )
)

## Diagnose the recorded demonstration ending

Restore the final saved pre-action state and execute only its
corresponding final recorded action.

This bypasses errors accumulated during full replay. It is a
state-restoration diagnostic, not learned-policy execution and
not evidence that full replay succeeds.

In [ ]:
assert task_name == "put_the_bowl_on_the_stove"

with h5py.File(dataset_path, "r") as demo_file:
    episode = demo_file["data"][episode_name]

    final_saved_state = episode["states"][-1].copy()
    final_recorded_action = episode["actions"][-1].copy()
    final_saved_image = episode["obs"]["agentview_rgb"][-1].copy()
    final_saved_joints = episode["obs"]["joint_states"][-1].copy()

diagnostic_env = None

try:
    diagnostic_env = OffScreenRenderEnv(
        bddl_file_name=str(bddl_path),
        camera_heights=128,
        camera_widths=128,
        control_freq=20,
        horizon=10,
        ignore_done=False,
    )

    diagnostic_env.seed(rollout_seed)
    diagnostic_env.reset()

    diagnostic_env.reset_from_xml_string(local_model_xml)
    diagnostic_env.sim.reset()

    before_observation = diagnostic_env.set_init_state(
        final_saved_state
    )

    success_before = bool(diagnostic_env.check_success())

    low, high = diagnostic_env.env.action_spec

    assert np.all(final_recorded_action >= np.asarray(low) - 1e-6)
    assert np.all(final_recorded_action <= np.asarray(high) + 1e-6)

    after_observation, _, _, _ = diagnostic_env.step(
        final_recorded_action.tolist()
    )

    success_after = bool(diagnostic_env.check_success())

    joint_error = float(
        np.max(
            np.abs(
                after_observation["robot0_joint_pos"]
                - final_saved_joints
            )
        )
    )

    print("Task:", task_name)
    print("Episode:", episode_name)
    print("Success before final action:", success_before)
    print("Success after final action:", success_after)
    print("Maximum final joint difference:", joint_error)

    comparison = np.concatenate(
        [
            final_saved_image[::-1],
            after_observation["agentview_image"][::-1],
        ],
        axis=1
    )

    print("Left: saved final image | Right: restored-and-stepped image")
    display(Image.fromarray(comparison))

finally:
    if diagnostic_env is not None:
        diagnostic_env.close()

    print("Diagnostic simulator closed.")

### Diagnostic finding

The restored final state of stove episode demo_34 satisfied the
simulator's success predicate both before and after its final action.

Full recorded-action replay from the earlier state did not succeed.
This supports accumulated replay drift as an explanation, but its
exact cause remains unresolved. State restoration does not reproduce
the full simulator/controller history.

The learned policy separately completed one stove development rollout
in 80 steps. Neither this restored-ending check nor that training-scene
rollout establishes held-out performance.